# Topic 30 · A character-level LSTM writes Shakespeare

CSCI 6379 · companion to the lecture note [Recurrent neural networks](https://dongchul.kim/csci6379-fall2026/topic/30).

The network reads text **one character at a time** and learns to predict the **next** character. Then we let it write: feed it a start, take its guess, feed the guess back in, and repeat. Watch the samples go from noise to words to something that looks like a play.

**Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. It takes about 5 minutes on a T4.

In [ ]:
import urllib.request, time
import torch, torch.nn as nn
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

# Tiny Shakespeare: 1.1 million characters of the plays (public domain), from Andrej Karpathy's char-rnn.
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urllib.request.urlopen(URL).read().decode("utf-8")
print(len(text), "characters. The start:\n")
print(text[:300])

## The data: every character becomes a number

In [ ]:
chars = sorted(set(text))                     # the vocabulary: every distinct character
V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}    # character -> id
itos = {i: c for c, i in stoi.items()}        # id -> character
data = torch.tensor([stoi[c] for c in text])
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]    # the last 10% of the text is held out
print(V, "different characters:", repr("".join(chars)))

SEQ, BATCH = 100, 64
def batch(split):
    # 64 random 100-character windows; the target is the same window shifted one character on
    d = train_data if split == "train" else val_data
    i = torch.randint(len(d) - SEQ - 1, (BATCH,))
    x = torch.stack([d[j:j + SEQ] for j in i])
    y = torch.stack([d[j + 1:j + SEQ + 1] for j in i])
    return x.to(device), y.to(device)

## The model: embedding, a two-layer LSTM, and a score for every character

This is **many-to-many**: after every character, a prediction of the next one.

In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, V, emb=64, hidden=256, layers=2):
        super().__init__()
        self.emb = nn.Embedding(V, emb)                      # character id -> 64 numbers
        self.lstm = nn.LSTM(emb, hidden, layers, batch_first=True)
        self.out = nn.Linear(hidden, V)                      # memory -> a score for each character

    def forward(self, x, state=None):
        h, state = self.lstm(self.emb(x), state)             # h: (batch, steps, hidden)
        return self.out(h), state                            # a prediction after every character

model = CharLSTM(V).to(device)
print(sum(p.numel() for p in model.parameters()), "weights")

@torch.no_grad()
def write(prompt="ROMEO:", n=300, temperature=0.8):
    # feed the prompt, then repeatedly sample the next character and feed it back in
    model.eval()
    x = torch.tensor([[stoi[c] for c in prompt]], device=device)
    logits, state = model(x)
    out = prompt
    for _ in range(n):
        p = torch.softmax(logits[0, -1] / temperature, dim=0)   # lower temperature = safer choices
        nxt = torch.multinomial(p, 1).item()
        out += itos[nxt]
        logits, state = model(torch.tensor([[nxt]], device=device), state)
    model.train()
    return out

print("Untrained, it writes noise:\n")
print(write(n=200))

## Training, with a sample after some of the epochs

In [ ]:
EPOCHS = 20
STEPS = len(train_data) // (SEQ * BATCH)       # one epoch = about one pass over the training text
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
loss_fn = nn.CrossEntropyLoss()
history = {"train": [], "val": []}
samples = {}
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    for _ in range(STEPS):
        x, y = batch("train")
        logits, _ = model(x)
        loss = loss_fn(logits.reshape(-1, V), y.reshape(-1))
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    with torch.no_grad():
        model.eval()
        val = sum(loss_fn(model(xv)[0].reshape(-1, V), yv.reshape(-1)).item()
                  for xv, yv in (batch("val") for _ in range(20))) / 20
        model.train()
    history["train"].append(loss.item()); history["val"].append(val)
    print(f"epoch {epoch:2d}  train loss {loss.item():.3f}  val loss {val:.3f}  ({time.time() - t0:.0f}s)")
    if epoch in (1, 2, 5, 10, 20):
        torch.manual_seed(1)
        samples[epoch] = write(n=200)
        print("-" * 60); print(samples[epoch]); print("-" * 60)

## How good is that? Compare with the simplest baseline

The baseline knows only how common each character is (spaces and 'e' are common, 'Z' is rare) and ignores what came before.

In [ ]:
counts = torch.bincount(train_data, minlength=V).float()
p = counts / counts.sum()
unigram = -(p[val_data] .log()).mean().item()           # the baseline's loss on the held-out text
print(f"baseline (character frequencies only): {unigram:.3f}   LSTM: {history['val'][-1]:.3f}")

plt.figure(figsize=(7, 3.5))
plt.plot(range(1, EPOCHS + 1), history["train"], label="LSTM, training")
plt.plot(range(1, EPOCHS + 1), history["val"], label="LSTM, held-out text")
plt.axhline(unigram, color="gray", ls="--", label="character frequencies only")
plt.xlabel("epoch"); plt.ylabel("loss (cross-entropy)"); plt.legend(); plt.show()

## Play with it

Change the prompt and the temperature. Low temperature (0.3) plays it safe and repeats itself; high (1.2) is adventurous and makes more mistakes.

In [ ]:
for t in (0.3, 0.8, 1.2):
    print(f"--- temperature {t} ---")
    print(write("JULIET:", n=250, temperature=t), "\n")